In [ ]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr as well as SNOMED categories
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
terms = list(set(x for lst in df_all["M_category"] for x in lst))

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary_new.csv"
df_feature_result = pd.read_csv(feature_result)

df_feature = df_feature_result[df_feature_result['status'] == "feature extraction complete"].copy()
df_feature = df_feature[df_feature['model']=='conch']["wsi_path"].astype(str)
with_features = list(set(df_feature))
print("WSIs with conch features detected: ", len(with_features))

In [ ]:
files = df_HE.loc[df_HE["filename"].isin(with_features), "filename"].tolist()
print("HE, with conch features extracted: ", len(files))

In [ ]:
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
import pandas as pd

df_with_features = df_HE[df_HE["filename"].isin(with_features)].copy()
df_single = df_with_features[df_with_features['M_category'].apply(len) == 1]
files = df_single["filename"].tolist()
print(len(files))

In [ ]:
from text_image_similarity import ClassifyMany

classifier = ClassifyMany(
    wsi_paths=files,
    local_zarr_dir=zarr_dir,
    classes=terms,
    dataset = df_single,
    model="conch",
    recompute=True
)
classifier.run_classification()

In [ ]:
import anndata as ad

agg_data = ad.read_h5ad("agg_conch_features.h5ad")

In [ ]:
print(agg_data.obs.columns.tolist())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=agg_data.obs,
    x="M_category",
    y="Normal Tissue",
    color="#C68FE6"
)
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Visualize Single Slide Classification
import os
from wsidata import open_wsi

path = files[0]
print(path)
zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path)
wsi

In [ ]:
from multimodal_analysis import SlideClassification
import lazyslide as zs

text_embeddings = zs.tl.text_embedding(terms, model="conch")

classifier = SlideClassification(wsi_path = path, local_zarr_dir = zarr_dir, classes = terms, model = "conch", text_embeddings = text_embeddings, recompute = False)

In [ ]:
slide_info = df_HE[df_HE["filename"] == path]

cols = ["team", "sex", "alder", "snomed_text", "T_category", "M_category"]
for col in cols: 
    print(col, slide_info[col].tolist())

In [ ]:
classifier.heatmap('Normal Tissue')

In [ ]:
classifier.all_heatmaps()